# Section 7 · Observability 2 — Local Tracing with Jaeger (Groq edition)

> **Groq edition.** Same lesson as the Bedrock notebook next to this one; only the model setup changes. Needs a `GROQ_API_KEY` (free at [console.groq.com/keys](https://console.groq.com/keys)) in your environment or a `.env` file.

Jaeger gives you a visual waterfall of each trace on your own machine.

Start Jaeger in a terminal first:

```bash
docker run -d --name jaeger \
  -e COLLECTOR_OTLP_ENABLED=true \
  -p 16686:16686 -p 4317:4317 -p 4318:4318 \
  jaegertracing/all-in-one:latest
```

Then open http://localhost:16686.

> Each tracing notebook should run in a **fresh kernel**: OpenTelemetry allows only one global tracer provider per Python process.

In [ ]:
%pip install -q -r requirements-groq.txt

In [ ]:
import os

from dotenv import load_dotenv
from strands.models.openai import OpenAIModel

load_dotenv()  # reads GROQ_API_KEY from a .env file, if present

MODEL_ID = os.getenv("GROQ_MODEL_ID", "openai/gpt-oss-120b")

model = OpenAIModel(
    client_args={
        "api_key": os.environ["GROQ_API_KEY"],
        "base_url": os.getenv("GROQ_BASE_URL", "https://api.groq.com/openai/v1"),  # or a local OpenAI-compatible server
        "max_retries": 5,  # retry dropped connections (the client default is 2)
    },
    model_id=MODEL_ID,
    # Groq reasoning models stream their thinking by default; hide it so only the answer prints.
    params={"extra_body": {"include_reasoning": False}},
)
print(f"Using {MODEL_ID} on Groq")

In [ ]:
from strands import Agent
from strands.telemetry import StrandsTelemetry
from strands_tools import calculator

# Send spans to Jaeger's OTLP/HTTP endpoint
os.environ["OTEL_EXPORTER_OTLP_ENDPOINT"] = "http://localhost:4318"
os.environ["OTEL_SERVICE_NAME"] = "strands-agents-demo"

strands_telemetry = StrandsTelemetry()
strands_telemetry.setup_otlp_exporter()

agent = Agent(model=model, system_prompt="You are a helpful assistant.", tools=[calculator])

response = agent("Explain the difference between traces, spans, and metrics in OpenTelemetry. Then compute 12% of 9,450.")

# Spans are sent in batches; flush before looking at Jaeger
strands_telemetry.tracer_provider.force_flush()

In Jaeger, choose the **strands-agents-demo** service and click **Find Traces**. The waterfall shows how long each model call took, tokens per call, the tool call with its inputs and output, and the total duration.

Stop Jaeger when you're done: `docker rm -f jaeger`